In [2]:
import os
import openai
os.environ["OPENAI_API_KEY"] = "" #모두의 연구소에서 발급
openai.api_key = os.getenv("OPENAI_API_KEY")

In [3]:
import json
from langchain.chat_models.openai import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.memory import ConversationSummaryBufferMemory
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder


def get_current_weather(lat, lon, unit="섭씨"):
    weather_info = {
        "lat": lat,
        "lon": lon,
        "temperature": "24",
        "unit": unit,
        "forecast": ["sunny", "windy"],
    }
    return json.dumps(weather_info)

user_function = {
    "name": "get_current_weather",
    "description": "주어진 지역의 현재 날씨를 알려줍니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "lat": {
                "type": "string",
                "description": "위도",
            },
            "lon": {
                "type": "string",
                "description": "경도",
            },
            "unit": {"type": "string", "enum": ["섭씨", "화씨"]},
        },
        "required": ["lat", "lon"],  # 수정된 부분
    },
}

chat_llm = ChatOpenAI(
    temperature=0.1,
).bind(
    function_call="auto",  # 수정된 부분
    functions=[user_function,]
)

memory_llm = ChatOpenAI(
    temperature=0.1,
)

memory = ConversationSummaryBufferMemory(
    llm=memory_llm,
    max_token_limit=50,
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a chatbot that helps people with their daily life."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

def load_memory(input):
    history = memory.load_memory_variables({})['history']
    return history

chain = RunnablePassthrough.assign(history=load_memory) | prompt | chat_llm

def invoke_chain(question):
    result = chain.invoke({"question": question})
    memory.save_context(
        {"input": question},
        {"output": result.content},
    )
    return result

**첫번째 질문 - 일반적인 질문**

In [4]:
response = invoke_chain("My name is Nico")
print(response)

content='Hello Nico! How can I assist you today?' response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 108, 'total_tokens': 119}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-554ee5da-c437-43a0-9e9e-d7a9fb8be095-0'


**첫번째 질문 - Function call에 관련된 질문**

In [5]:
response = invoke_chain("오늘 을지로 날씨 어때?")
print(response)

content='' additional_kwargs={'function_call': {'arguments': '{"lat":"37.5665","lon":"126.978","unit":"섭씨"}', 'name': 'get_current_weather'}} response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 143, 'total_tokens': 175}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'function_call', 'logprobs': None} id='run-17f91595-7666-4b36-a933-3de02a85f0fe-0'


In [6]:
def function_calling(input):
    available_functions = {
            "get_current_weather": get_current_weather,
        }

    function_call = input.additional_kwargs["function_call"]
    function_name = function_call["name"]
    function_to_call = available_functions.get(function_name)

    if function_to_call:
        # Extract arguments from the function call
        function_args = json.loads(function_call["arguments"])
        function_input = function_to_call(
            lat=function_args.get("lat"),
            lon=function_args.get("lon"),
            unit=function_args.get("unit", "섭씨")
        )

        result = invoke_chain(function_input)

        return result.content

    return "Not exist function call"

if response.additional_kwargs.get("function_call"):
    result = function_calling(response)

print(result)

을지로의 오늘 날씨는 맑고 바람이 많이 불 것으로 예상됩니다. 온도는 24°C입니다. 혹시 더 궁금한 사항이 있나요?
